In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import json
import time
from agentlib.trace import Tracer
from agentlib.tools import registry
from agentlib.llm import get_client
from agentlib.tools import get_concept_and_descendant_synonyms
from source_data_config import BasicConfig
from agentlib.extract_snomed import extract_terms, get_concept, resolve_ids

from agentlib.logic import extract_logic, split_routes, expand_routes, tree_to_text



import polars as pl
client, model = get_client(require_local=True)   # returns (client, model name); HERO: local only

# 0. build fake query text

In [1]:
concepts = pl.read_parquet(BasicConfig().concept_path)

text_body = """
    I want papers that study ('resilience' or 'protective mechanism')
    for 'solid organ transplant patients'[313039003] 
    that report at least one of these laboratory measurements: 
    ('C-reactive protein'[55235003] or 'albumin' [999997] 
    or 'interleukin-6' or 'neutrophil-to-lymphocyte ratio' 
    or 'lymphocyte count' or 'hemoglobin' or 'cortisol').
    Please exclude papers that study 'pediatric' or 'children' or 'adolescent' or 'teenager'.
    """



NameError: name 'pl' is not defined

# 1.  Extract terms and IDs

In [ ]:
terms  = extract_terms(text_body)

# 2. Extract the logic

In [ ]:
logic  = extract_logic(client, model, text_body)      # LLM, checked by code, up to 2 retries
routes = split_routes(terms)                          # c5 -> c5t and c5i
tree   = expand_routes(logic["tree"], routes)         # c5 -> {"or": ["c5t", "c5i"]}

labels = {t["key"]: t["term"] for t in terms}
print(tree_to_text(logic["tree"], labels))            # the sentence you confirm

NameError: name 'extract_logic' is not defined

# 3. check validation on id

In [10]:
hi = resolve_ids(terms, interactive=True)
hi

"solid organ transplant patients"  <-  Solid organ transplant to recipient (procedure)
"C-reactive protein"  <-  C-reactive protein measurement (procedure)


: 

: 

# 4. Find the MeSH descriptors

In [20]:
# Find the MeSH routed by ID
import json, os, urllib.parse, urllib.request, urllib.error

UTS = "https://uts-ws.nlm.nih.gov/rest"
API_KEY = os.getenv("UMLS_API_KEY", "")

def _get_json(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.load(r)

def mesh_by_id(snomed_id):
    """Route by ID: SNOMED CT ID -> MeSH descriptors that share a UMLS concept."""
    found, page = [], 1
    while True:
        q = urllib.parse.urlencode({"targetSource": "MSH", "apiKey": API_KEY, "pageNumber": page})
        try:
            body = _get_json(f"{UTS}/crosswalk/current/source/SNOMEDCT_US/{snomed_id}?{q}")
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return []
            raise
        for item in body.get("result", []):
            if item.get("rootSource") == "MSH" and item["ui"].startswith("D"):
                found.append(item["ui"])
        if page >= body.get("pageCount", 1):
            return found
        page += 1

mesh_by_id(D016030)

NameError: name 'D016030' is not defined

In [13]:
import difflib

LOOKUP = "https://id.nlm.nih.gov/mesh/lookup"

def mesh_by_term(term, match="contains", limit=20):
    """Route by name: candidate MeSH descriptors whose label or entry term contains the text."""
    q = urllib.parse.urlencode({"label": term, "match": match, "limit": limit})
    items = _get_json(f"{LOOKUP}/descriptor?{q}")
    return [{"label": i["label"], "ui": i["resource"].rsplit("/", 1)[-1]} for i in items]

def rank_by_similarity(term, candidates):
    """Typo-tolerant order: most similar label first. Your decision still picks the descriptor."""
    key = lambda c: difflib.SequenceMatcher(None, term.lower(), c["label"].lower()).ratio()
    return sorted(candidates, key=key, reverse=True)

hits = mesh_by_term("albumIn", match="contains")     # misspelled on purpose
for c in rank_by_similarity("albumn", hits)[:5]:
    print(c)

{'label': 'Albumins', 'ui': 'D000418'}
{'label': 'Albuminuria', 'ui': 'D000419'}
{'label': 'Serum Albumin', 'ui': 'D012709'}
{'label': '2S Albumins, Plant', 'ui': 'D055516'}
{'label': 'Receptors, Albumin', 'ui': 'D017457'}


In [ ]:
def debug_crosswalk(snomed_id):
    q = urllib.parse.urlencode({"targetSource": "MSH", "apiKey": API_KEY})
    body = _get_json(f"{UTS}/crosswalk/curren
                     t/source/SNOMEDCT_US/{snomed_id}?{q}")
    print(json.dumps(body, indent=1)[:3000])

debug_crosswalk("70536003")

{
 "pageSize": 25,
 "pageNumber": 1,
 "pageCount": 1,
 "result": [
  {
   "classType": "SourceAtomCluster",
   "ui": "D016030",
   "suppressible": false,
   "obsolete": false,
   "rootSource": "MSH",
   "atomCount": 14,
   "cVMemberCount": 0,
   "attributes": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030/attributes",
   "atoms": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030/atoms",
   "ancestors": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030/ancestors",
   "parents": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030/parents",
   "children": "NONE",
   "descendants": "NONE",
   "relations": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030/relations",
   "definitions": "NONE",
   "concepts": "https://uts-ws.nlm.nih.gov/rest/search/2026AB?string=D016030&sabs=MSH&searchType=exact&inputType=sourceUi",
   "defaultPreferredAtom": "https://uts-ws.nlm.nih.gov/rest/content/2026AB/source/MSH/D016030